# Score-sizing denominator · frozen IS mean(|z|) vs rolling mean(|z|)

Other test (not a STAR change). Compares the **sealed backtest / tearsheet** sizing path
(one IS-frozen book-level mean of `|z|`) to the **paper runner** path (per-pair PIT rolling
mean of `|z|`).

| Arm | Denominator | Matches |
|-----|-------------|---------|
| `frozen_is` | `fit_mean_abs_score` on research IS | `01_star_tearsheet` / H-012 sealed backtest |
| `rolling_{w}` | per-pair rolling mean `|z|`, window `w` | `walk_live_book` / paper runner |

STAR stack unchanged. `SIZE_STAR=score` throughout.

**Helpers:** `04_backtest/s2_coint/diagnosis.py`

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.diagnosis import (
    mean_abs_score_summary,
    plot_rolling_mean_abs,
    run_score_denom_backtest,
)
from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    frozen_pairs_for_universe,
    is_end_for_stack,
    load_s1_weekly,
    load_universe_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    split_is_oos,
)
from backtest.s2_coint.tearsheet import fit_mean_abs_score

# Adaptable rolling lookbacks for the sizing denominator (paper runner uses Z_WINDOW_STAR).
ROLLING_WINDOWS: list[int] = [60, 90]

USE_OUT_OF_SAMPLE_DATA = False
STAR_PATH = DEFAULT_STAR_STACK
stack = load_star_stack(STAR_PATH)
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
require_star("SIZE_STAR", stack.get("SIZE_STAR"))
UNIVERSE = str(stack["UNIVERSE_STAR"])
PAIRS = list(stack.get("PAIRS_STAR") or frozen_pairs_for_universe(UNIVERSE, "1d", root=ROOT))
Z_WINDOW_STAR = int(stack.get("Z_WINDOW_STAR") or 90)

print("UNIVERSE", UNIVERSE)
print("PAIRS", PAIRS)
print("SIZE_STAR", stack.get("SIZE_STAR"))
print("Z_WINDOW_STAR", Z_WINDOW_STAR)
print("ROLLING_WINDOWS", ROLLING_WINDOWS)

## 1. Load panel and overlay STAR hedge / z columns

In [ ]:
require_star("BAR_STAR", stack.get("BAR_STAR"))
BAR = str(stack["BAR_STAR"])
train, full = load_universe_panels(UNIVERSE, BAR, PAIRS, root=ROOT)
lb = lookbacks_for_bar(
    BAR,
    ols_days=int(stack["OLS_WINDOW_STAR"]) if stack.get("OLS_WINDOW_STAR") not in (None, "") else None,
    z_days=int(stack["Z_WINDOW_STAR"]) if stack.get("Z_WINDOW_STAR") not in (None, "") else None,
    adf_days=int(stack["ADF_WINDOW_STAR"]) if stack.get("ADF_WINDOW_STAR") not in (None, "") else None,
)
train = overlay_ols_hedge(
    train,
    ols_window=lb["ols_window"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
    adf_window=lb["adf_window"],
)
full = overlay_ols_hedge(
    full,
    ols_window=lb["ols_window"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
    adf_window=lb["adf_window"],
)
if stack.get("HEDGE_STAR") == "kalman":
    kf_delta = stack["KALMAN_DELTA_STAR"]
    train = overlay_kalman_hedge(
        train,
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
        delta=kf_delta,
    )
    full = overlay_kalman_hedge(
        full,
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
        delta=kf_delta,
    )

is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
panel = full
work = panel if USE_OUT_OF_SAMPLE_DATA else is_panel
s1_weekly = load_s1_weekly(ROOT)

frozen_scalar = fit_mean_abs_score(work, date_mask=pd.to_datetime(work["date"]) <= is_end)
print("bar", BAR, "is_end", is_end)
print("work rows", len(work), "pairs", work["pair_id"].nunique())
print("book-level frozen IS mean(|z|)", round(frozen_scalar, 4))
work.head()

## 2. Mean |z| summary by pair and window

Per pair: full-sample and IS-only simple means of `|z|`, plus rolling-window stats.
`frozen_is_scalar` is the **pooled** book constant used by the sealed backtest.

In [ ]:
summary = mean_abs_score_summary(
    work,
    ROLLING_WINDOWS,
    score_column="z",
    is_end=is_end,
)
display(summary)

pivot = summary.loc[summary["window"] != "frozen_is"].pivot_table(
    index="pair_id",
    columns="window",
    values="rolling_mean_full",
    aggfunc="first",
)
pivot.columns = [f"rolling_mean_full_w{int(c)}" for c in pivot.columns]
pivot["frozen_is_scalar"] = frozen_scalar
pivot["mean_abs_z_is"] = (
    summary.loc[summary["window"] == "frozen_is"]
    .drop_duplicates("pair_id")
    .set_index("pair_id")["mean_abs_z_is"]
)
print("\nPer-pair rolling full-sample mean of rolling mean(|z|) vs frozen IS scalar:")
display(pivot.round(4))

## 3. Rolling mean |z| time series

Dashed line = pooled IS-frozen scalar (`fit_mean_abs_score`). Solid lines = PIT rolling
mean `|z|` for each window in `ROLLING_WINDOWS`.

In [ ]:
plot_rolling_mean_abs(
    work,
    ROLLING_WINDOWS,
    score_column="z",
    pairs=PAIRS,
    is_end=is_end,
    frozen_is_scalar=frozen_scalar,
)

## 4. Backtest arms · frozen vs rolling denominators

Full frozen STAR stack (`run_s2_backtest`). Research IS only unless
`USE_OUT_OF_SAMPLE_DATA = True`.

In [ ]:
arms: dict[str, dict] = {
    "frozen_is": {"denom_mode": "frozen_is", "rolling_window": None},
}
for w in ROLLING_WINDOWS:
    arms[f"rolling_{w}"] = {"denom_mode": "rolling", "rolling_window": int(w)}

rows = []
results = {}
for arm, kw in arms.items():
    res = run_score_denom_backtest(
        panel,
        stack,
        s1_weekly=s1_weekly,
        is_end=is_end,
        use_oos=USE_OUT_OF_SAMPLE_DATA,
        **kw,
    )
    results[arm] = res
    row = {"arm": arm, **res.metrics}
    rows.append(row)
    print(arm, res.metrics)

metrics_df = pd.DataFrame(rows).set_index("arm")
metrics_df

## 5. Arm comparison (primary metrics)

In [ ]:
cols = ["ann_sharpe", "max_drawdown", "corr_to_s1", "n_days"]
display(metrics_df[cols].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
metrics_df["ann_sharpe"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Research IS · ann Sharpe by score-denom arm")
ax.set_ylabel("ann_sharpe")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

if not USE_OUT_OF_SAMPLE_DATA:
    print(
        "Note: rolling live scale was not walk-forwarded in STAR selection. "
        "Set USE_OUT_OF_SAMPLE_DATA=True for sealed OOS once arms are reviewed."
    )